# 03 · Jev 호환 HTTP API 테스트
**환경:** Python 3.11+ 표준 라이브러리, 로컬 Unsloth Studio 서버 (Laya 또는 02에서 학습한 모델).
**목표:** 요청/응답 계약, 실제 추론, cold/warm 지연시간, 소규모 라벨 평가와 보류 정책.

기본 `LIVE_API=False`에서는 **합성 fixture 계약 검사만** 수행한다. 실제 모델을 호출하려면 서버를 켜고 True로 바꾼다.
서버가 없을 때 가짜 추론 결과로 대체하지 않는다. 이 파일 하나만 옮겨 실행할 수 있다.

### 준비
1. Unsloth Studio → Settings → API → Decision API → Serve requests.
2. Laya multilingual 또는 02의 merged 모델을 선택한다.
3. Access token을 환경변수 `TYPESAFE_API_KEY`에 설정한다. 노트북에 키를 저장하지 않는다.
4. GPU 서버가 원격이면 SSH 터널을 열고 base URL은 localhost로 둔다.

02의 모델을 쓰려면 GPU 서버에서:
```bash
UNSLOTH_SYSTEMONE_MODEL=/absolute/path/to/merged unsloth studio -H 127.0.0.1 -p 8888
```
`jev-latest`는 이 로컬 서버에서 선택 모델에 대한 별칭이다. 실제 TypeSafe Jev 호출이라는 뜻이 아니다.


In [ ]:
import json, math, os, time
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

def validate_answers(response, questions):
    answers = response.get('answers')
    if not isinstance(answers, dict) or set(answers) != set(questions):
        raise ValueError('missing or unexpected answer keys')
    def probability(value):
        if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError('probability must be finite and in [0,1]')
    for name, question in questions.items():
        answer = answers[name]
        kind = question['type']
        if answer.get('type') != kind:
            raise ValueError(f'{name}: wrong answer type')
        if kind == 'noul':
            probability(answer['noul'])
            continue
        probs = answer['probabilities']
        expected = set(question['criteria']) if kind == 'choice' else {str(i) for i in range(len(question['criteria']))}
        if set(probs) != expected:
            raise ValueError(f'{name}: wrong probability keys')
        for p in probs.values():
            probability(p)
        if not math.isclose(sum(probs.values()), 1.0, abs_tol=1e-3):
            raise ValueError(f'{name}: probabilities do not sum to one')
        if kind == 'choice':
            chosen = answer['choice']
            if chosen not in probs or probs[chosen] < max(probs.values()) - 1e-6:
                raise ValueError(f'{name}: choice does not match argmax')
        elif kind == 'score':
            score = answer['score']
            expected_score = sum(int(k) * p for k, p in probs.items())
            if not isinstance(score, (int, float)) or not math.isfinite(score) or not math.isclose(score, expected_score, abs_tol=1e-3):
                raise ValueError(f'{name}: score is not the expected level')
        else:
            raise ValueError(f'unsupported question type: {kind}')
    return answers

def request_decision(base_url, model, state, questions, api_key='', timeout=180):
    body = json.dumps({'model':model, 'state':state, 'questions':questions}, ensure_ascii=False).encode()
    headers = {'Content-Type':'application/json'}
    if api_key:
        headers['Authorization'] = 'Bearer ' + api_key
    request = Request(base_url.rstrip('/') + '/v1/systemone', data=body, headers=headers, method='POST')
    started = time.perf_counter()
    try:
        with urlopen(request, timeout=timeout) as result:
            response = json.load(result)
    except HTTPError as exc:
        raise RuntimeError(f'HTTP {exc.code}: API 활성화, 모델 및 access token을 확인하세요.') from exc
    except (URLError, TimeoutError) as exc:
        raise RuntimeError('서버 주소/SSH 터널/모델 로딩 시간/timeout을 확인하세요.') from exc
    elapsed = time.perf_counter() - started
    validate_answers(response, questions)
    return response, elapsed


In [ ]:
LIVE_API = False
BASE_URL = os.environ.get('TYPESAFE_BASE_URL', 'http://127.0.0.1:8888')
API_KEY = os.environ.get('TYPESAFE_API_KEY', '')
MODEL = 'jev-latest'
questions = {
    'team': {'type':'choice', 'instructions':'담당 팀은?',
             'criteria':{'billing':'결제와 환불', 'technical':'장애와 오류', 'sales':'가격과 계약'}},
    'refund': {'type':'noul', 'instructions':'환불 요청이 있는가?'},
    'urgency': {'type':'score', 'instructions':'긴급도는?', 'criteria':['일반','곧 처리','오늘 처리']},
}
state = '결제가 두 번 됐어요. 오늘 중복 금액을 환불해 주세요.'
print({'mode': 'LIVE MODEL' if LIVE_API else 'OFFLINE CONTRACT CHECK ONLY', 'base_url':BASE_URL, 'model_alias':MODEL})


## 1. 오프라인 계약 검사
fixture는 직접 작성한 교육용 확률이다. 합계, 선택 키, score 기댓값과 noul 범위를 검사한다.
실제 모델 정확도와는 관련 없다. confidence는 공급자별 의미가 달라 라우팅 계산에 쓰지 않는다.


In [ ]:
fixture = {'model':'synthetic-fixture-NOT-A-MODEL', 'answers':{
    'team': {'type':'choice','choice':'billing','probabilities':{'billing':.8,'technical':.15,'sales':.05}},
    'refund': {'type':'noul','noul':.9},
    'urgency': {'type':'score','score':1.6,'probabilities':{'0':.1,'1':.2,'2':.7}},
}}
validate_answers(fixture, questions)
print('합성 응답 계약 검사 통과. 실제 추론은 아직 수행하지 않았습니다.')


## 2. 실제 한 번의 요청
첫 요청은 모델 로딩 시간을 포함할 수 있다. 서버가 이미 준비된 상태라면 cold-start 측정은 아니다.
HTTP 에러는 그대로 실패시킨다. 선택된 모델 이름은 응답에서 확인한다.


In [ ]:
if LIVE_API:
    response, first_latency = request_decision(BASE_URL, MODEL, state, questions, API_KEY)
    print('first request seconds:', round(first_latency, 3))
    print(json.dumps(response, ensure_ascii=False, indent=2))
else:
    print('SKIPPED: 실제 추론을 하려면 LIVE_API=True로 변경하세요.')


## 3. Warm latency와 소규모 정확도
아래 직접 작성한 한국어 문의는 세 팀마다 두 건이다. 통계적 벤치마크가 아니라 연결/동작 확인용이다.
측정은 client end-to-end 순차 요청 기준이고 서버 큐/네트워크/토큰화를 포함한다.
같은 입력 반복은 캐시 영향을 받을 수 있다. 공식 성능 수치와 직접 비교하지 않는다.


In [ ]:
cases = [
    ('중복 청구된 금액을 환불해 주세요.', 'billing'),
    ('이번 달 청구서 금액이 잘못됐습니다.', 'billing'),
    ('로그인하면 오류 코드가 나오고 접속이 안 됩니다.', 'technical'),
    ('업데이트 후 앱이 계속 종료됩니다.', 'technical'),
    ('기업용 요금제와 계약 조건을 알려 주세요.', 'sales'),
    ('팀 계정 50개의 신규 견적을 받고 싶습니다.', 'sales'),
]
if LIVE_API:
    import statistics
    records = []
    for text, gold in cases:
        result, seconds = request_decision(BASE_URL, MODEL, text, questions, API_KEY)
        answer = result['answers']['team']
        records.append({'gold':gold, 'prediction':answer['choice'],
                        'probabilities':answer['probabilities'], 'latency_s':seconds,
                        'served_model':result.get('model')})
    ordered = sorted(r['latency_s'] for r in records)
    summary = {'n':len(records), 'accuracy':sum(r['gold']==r['prediction'] for r in records)/len(records),
               'p50_s':statistics.median(ordered), 'p95_nearest_rank_s':ordered[math.ceil(.95*len(ordered))-1]}
    print(summary)
    print('6건의 p95는 사실상 최대값이므로 안정적인 성능 지표가 아닙니다.')
    from pathlib import Path
    from datetime import datetime, timezone
    output = Path('artifacts') / ('api-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
    output.parent.mkdir(exist_ok=True)
    output.write_text(json.dumps({'summary':summary,'records':records}, ensure_ascii=False, indent=2))
    print('saved:', output.resolve())
else:
    print('SKIPPED: 라이브 라벨 평가와 지연시간 측정')


## 4. 보류 정책과 연습
여기서 .85는 정책 흐름 설명용이다. 실제 기준값은 별도의 validation으로 선택하고 test에서는 고정한다.
이 예제는 라우팅 결과를 출력할 뿐 환불 등의 외부 작업을 실행하지 않는다.

**문제:** '환불은 원하지 않고 오류만 고쳐 주세요'를 추가하고 noul과 team이 어떻게 달라지는지 확인하자.
질문이나 선택지 설명을 바꿀 때 동일한 test 사례로 회귀 검증한다.


In [ ]:
def route(answer, threshold=.85):
    selected = answer['choice']
    return selected if answer['probabilities'][selected] >= threshold else 'human_review'

if LIVE_API:
    print('route:', route(response['answers']['team']))
    extra, _ = request_decision(BASE_URL, MODEL, '환불은 원하지 않고 오류만 고쳐 주세요.', questions, API_KEY)
    print('연습 관찰:', extra['answers'])
else:
    print('합성 fixture 정책 결과:', route(fixture['answers']['team']))
    assert route(fixture['answers']['team']) == 'human_review'


## 해석과 문제 해결
- connection refused: Studio 실행, 포트, SSH 터널을 확인한다.
- 401/403: Access token 또는 localhost keyless 설정을 확인한다.
- timeout: 최초 모델 로딩/GPU 점유를 확인하고 timeout을 늘린다.
- 선택지 누락/NaN/합계 오류: 응답 계약 실패다. 임의 보정으로 숨기지 않는다.
- 한국어 정확도 저하: multilingual 모델 선택과 별도 한국어 평가 데이터를 확인한다.
- 02로 학습한 모델을 테스트하려면 merged 경로 및 Studio 선택 모델을 확인한다.

확장 과제: 실제 업무의 독립 test 파일로 cases를 교체하고 팀별 confusion matrix,
보류 비율, 유형별 정확도 및 latency 분포를 보고하자.

출처: [Unsloth 로컬 Decision API](https://unsloth.ai/docs/models/decision-laya),
[TypeSafe Jev 소개](https://typesafe.ai/blog/introducing-system-one-models-and-jev).
